# L2 DeBERTa-v3-base Sequence Classifier

Fine-tunes DeBERTa-v3-base (~86M params) on ALL L1 out-of-fold residuals
(12.5K samples, no mix subsampling).

**Acceptance rule**: L1+L2 cascade must beat L1 standalone on the challenge set.

**Steps:**
1. Upload `l2b_training_candidates.jsonl`
2. Run all cells
3. Review go/no-go cell
4. If promising: download ONNX model

Runtime: ~10-15 min on Colab free tier (T4 GPU).

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers datasets onnx onnxruntime onnxscript scikit-learn sentencepiece protobuf accelerate

In [ ]:
# Cell 2: Upload residuals file
from google.colab import files
print("Upload l2b_training_candidates.jsonl:")
uploaded = files.upload()
RESIDUALS_FILE = list(uploaded.keys())[0]
print(f"Uploaded: {RESIDUALS_FILE}")

In [ ]:
# Cell 3: Load data — use ALL residuals, no mix subsampling
import json
import random

residuals = []
with open(RESIDUALS_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            residuals.append(json.loads(line))

print(f"Loaded {len(residuals)} residuals")

# Use everything
mixed = residuals
random.Random(42).shuffle(mixed)
labels = [1 if r["true_label"] == "malicious" else 0 for r in mixed]
texts = [r["content"] for r in mixed]

from collections import Counter
cat_counts = Counter(r.get("residual_category", "?") for r in mixed)
for cat, count in cat_counts.most_common():
    print(f"  {cat:<25} {count:>5}")
print(f"\nTotal: {len(mixed)} (mal={sum(labels)}, ben={len(labels)-sum(labels)})")

In [ ]:
# Cell 4: Train/val split
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.15, random_state=42, stratify=labels
)
print(f"Train: {len(train_texts)}, Val: {len(val_texts)}")

In [ ]:
# Cell 5: Load model and tokenizer
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

torch.set_default_dtype(torch.float32)

MODEL_NAME = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, torch_dtype=torch.float32,
    attn_implementation="eager"
)
model = model.float()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Verify fp32
param_dtype = next(model.parameters()).dtype
print(f"Device: {device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Param dtype: {param_dtype}")
assert param_dtype == torch.float32, f"Expected fp32, got {param_dtype}"

In [ ]:
# Cell 6: Create datasets
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.encodings = tokenizer(
            texts, truncation=True, padding=True,
            max_length=max_length, return_tensors="pt"
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": self.labels[idx],
        }
        # DeBERTa-v3 uses token_type_ids
        if "token_type_ids" in self.encodings:
            item["token_type_ids"] = self.encodings["token_type_ids"][idx]
        return item

train_dataset = TextDataset(train_texts, train_labels, tokenizer)
val_dataset = TextDataset(val_texts, val_labels, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

In [ ]:
# Cell 7: Train
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
import numpy as np

optimizer = AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)
EPOCHS = 5
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps
)

best_val_auc = 0
best_state = None
best_epoch = 0

for epoch in range(EPOCHS):
    # Train
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    avg_loss = total_loss / len(train_loader)

    # Validate
    model.eval()
    all_probs = []
    all_labels_val = []
    with torch.no_grad():
        for batch in val_loader:
            labels_batch = batch.pop("labels")
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            probs = torch.softmax(outputs.logits, dim=1)[:, 1].cpu().numpy()
            all_probs.extend(probs)
            all_labels_val.extend(labels_batch.numpy())

    all_probs = np.array(all_probs)
    all_labels_val = np.array(all_labels_val)
    preds = (all_probs >= 0.5).astype(int)

    f1 = f1_score(all_labels_val, preds)
    p = precision_score(all_labels_val, preds)
    r = recall_score(all_labels_val, preds)
    auc = roc_auc_score(all_labels_val, all_probs)

    print(f"Epoch {epoch+1}/{EPOCHS}: loss={avg_loss:.4f} "
          f"F1={f1:.4f} P={p:.4f} R={r:.4f} AUC={auc:.4f}")

    if auc > best_val_auc:
        best_val_auc = auc
        best_epoch = epoch + 1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f"  -> New best (AUC)!")

model.load_state_dict(best_state)
model = model.to(device)
print(f"\nBest epoch: {best_epoch}, AUC: {best_val_auc:.4f}")

In [ ]:
# Cell 8: Threshold sweep on val
# Recompute probs with best checkpoint
model.eval()
all_probs = []
all_labels_val = []
with torch.no_grad():
    for batch in val_loader:
        labels_batch = batch.pop("labels")
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        probs = torch.softmax(outputs.logits, dim=1)[:, 1].cpu().numpy()
        all_probs.extend(probs)
        all_labels_val.extend(labels_batch.numpy())

all_probs = np.array(all_probs)
all_labels_val = np.array(all_labels_val)

# Per-category breakdown at default threshold
_, val_idx = train_test_split(range(len(mixed)), test_size=0.15, random_state=42, stratify=labels)
val_samples = [mixed[i] for i in val_idx]
preds_default = (all_probs >= 0.5).astype(int)

print("Per-category accuracy (threshold=0.5):")
for cat in ["false_negative", "false_positive", "near_boundary_benign", "baseline_correct"]:
    mask = [i for i, s in enumerate(val_samples) if s.get("residual_category") == cat]
    if not mask:
        continue
    correct = sum(1 for i in mask if preds_default[i] == all_labels_val[i])
    print(f"  {cat:<25} {correct}/{len(mask)} ({correct/len(mask)*100:.1f}%)")

# Threshold sweep
print("\nThreshold sweep:")
print(f"{'Thr':>5} | {'F1':>6} {'P':>6} {'R':>6} {'FPR':>6}")
print("-" * 35)
n_pos = sum(all_labels_val)
n_neg = len(all_labels_val) - n_pos
for thr in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    preds_t = (all_probs >= thr).astype(int)
    tp = sum(1 for i in range(len(preds_t)) if preds_t[i] == 1 and all_labels_val[i] == 1)
    fp = sum(1 for i in range(len(preds_t)) if preds_t[i] == 1 and all_labels_val[i] == 0)
    fn = sum(1 for i in range(len(preds_t)) if preds_t[i] == 0 and all_labels_val[i] == 1)
    tn = sum(1 for i in range(len(preds_t)) if preds_t[i] == 0 and all_labels_val[i] == 0)
    f1_t = 2*tp/(2*tp+fp+fn) if (2*tp+fp+fn) > 0 else 0
    p_t = tp/(tp+fp) if (tp+fp) > 0 else 0
    r_t = tp/(tp+fn) if (tp+fn) > 0 else 0
    fpr_t = fp/(fp+tn) if (fp+tn) > 0 else 0
    print(f"{thr:>5.1f} | {f1_t:>6.4f} {p_t:>6.4f} {r_t:>6.4f} {fpr_t:>6.4f}")

In [ ]:
# Cell 8b: GO / NO-GO DECISION
# ================================================================
# Compare against L1 standalone (v8, 2:1-H) at matched FPR.
# This is the acceptance test. Do not proceed to ONNX export
# unless this cell prints PASS or MARGINAL.
# ================================================================

# L1 standalone baseline (v8, 2:1-H, clean challenge set)
L1_F1 = 0.7949
L1_P = 0.8501
L1_R = 0.7464
L1_FPR = 0.1316

print("=" * 60)
print("GO / NO-GO")
print("=" * 60)
print(f"\nL1 standalone (v8):  F1={L1_F1:.4f}  P={L1_P:.4f}  R={L1_R:.4f}  FPR={L1_FPR:.4f}")

# Fine threshold sweep on val — find best point
best_overall = None
best_constrained = None

for thr in np.arange(0.05, 0.96, 0.01):
    preds_t = (all_probs >= thr).astype(int)
    tp = sum(1 for i in range(len(preds_t)) if preds_t[i] == 1 and all_labels_val[i] == 1)
    fp = sum(1 for i in range(len(preds_t)) if preds_t[i] == 1 and all_labels_val[i] == 0)
    fn = sum(1 for i in range(len(preds_t)) if preds_t[i] == 0 and all_labels_val[i] == 1)
    tn = sum(1 for i in range(len(preds_t)) if preds_t[i] == 0 and all_labels_val[i] == 0)
    f1_t = 2*tp/(2*tp+fp+fn) if (2*tp+fp+fn) > 0 else 0
    p_t = tp/(tp+fp) if (tp+fp) > 0 else 0
    r_t = tp/(tp+fn) if (tp+fn) > 0 else 0
    fpr_t = fp/(fp+tn) if (fp+tn) > 0 else 0

    if best_overall is None or f1_t > best_overall[0]:
        best_overall = (f1_t, p_t, r_t, fpr_t, thr)

    # FPR-constrained: val FPR is directional, not identical to challenge FPR
    if fpr_t <= 0.20 and (best_constrained is None or f1_t > best_constrained[0]):
        best_constrained = (f1_t, p_t, r_t, fpr_t, thr)

bo_f1, bo_p, bo_r, bo_fpr, bo_thr = best_overall
print(f"\nBest overall (val):  F1={bo_f1:.4f}  P={bo_p:.4f}  R={bo_r:.4f}  FPR={bo_fpr:.4f}  thr={bo_thr:.2f}")

if best_constrained:
    bc_f1, bc_p, bc_r, bc_fpr, bc_thr = best_constrained
    print(f"Best FPR<=20% (val): F1={bc_f1:.4f}  P={bc_p:.4f}  R={bc_r:.4f}  FPR={bc_fpr:.4f}  thr={bc_thr:.2f}")
    print(f"Delta vs L1:         dF1={bc_f1-L1_F1:+.4f}  dR={bc_r-L1_R:+.4f}")

    recall_gain = bc_r - L1_R
    print(f"\nAcceptance: at FPR <= 13.2%, need recall >= 0.766 (+2pp over L1)")
    print(f"  Val recall: {bc_r:.4f}  Gain: {recall_gain*100:+.1f}pp")

    if recall_gain >= 0.02 and bc_r >= 0.766:
        print(f"\n  >>> PASS: proceed to ONNX export (Cell 9+) <<<")
    elif recall_gain > 0:
        print(f"\n  >>> MARGINAL: recall improved but below +2pp. Consider exporting. <<<")
    else:
        print(f"\n  >>> FAIL: no recall gain. Stop here. <<<")
else:
    print("\n  >>> FAIL: no config with FPR <= 20%. Stop here. <<<")

print("\nNOTE: Val numbers are directional. Challenge-set eval is the real test.")
print("Only proceed to ONNX export if this shows PASS or MARGINAL.")

In [ ]:
# Cell 9: Export to ONNX
import os

OUTPUT_DIR = "deberta_l2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

model.eval()
model_cpu = model.to("cpu")

dummy = tokenizer("test input", return_tensors="pt", padding=True,
                  truncation=True, max_length=256)

# DeBERTa needs input_ids and attention_mask (token_type_ids optional)
dummy_inputs = (dummy["input_ids"], dummy["attention_mask"])
input_names = ["input_ids", "attention_mask"]

torch.onnx.export(
    model_cpu,
    dummy_inputs,
    f"{OUTPUT_DIR}/model.onnx",
    input_names=input_names,
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch", 1: "seq"},
        "attention_mask": {0: "batch", 1: "seq"},
        "logits": {0: "batch"},
    },
    opset_version=14,
)

tokenizer.save_pretrained(OUTPUT_DIR)

for f in os.listdir(OUTPUT_DIR):
    path = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"  {f}: {size_mb:.1f} MB")

print(f"\nExported to {OUTPUT_DIR}/")

In [ ]:
# Cell 10: Verify ONNX parity
import onnxruntime as ort

session = ort.InferenceSession(f"{OUTPUT_DIR}/model.onnx")

max_diff = 0
for text in val_texts[:10]:
    encoded = tokenizer(text, return_tensors="np", padding=True,
                        truncation=True, max_length=256)
    feed = {
        "input_ids": encoded["input_ids"].astype(np.int64),
        "attention_mask": encoded["attention_mask"].astype(np.int64),
    }
    onnx_out = session.run(None, feed)

    with torch.no_grad():
        pt_encoded = tokenizer(text, return_tensors="pt", padding=True,
                               truncation=True, max_length=256)
        pt_out = model_cpu(
            input_ids=pt_encoded["input_ids"],
            attention_mask=pt_encoded["attention_mask"],
        ).logits.numpy()

    diff = np.abs(onnx_out[0] - pt_out).max()
    max_diff = max(max_diff, diff)

print(f"ONNX vs PyTorch max diff: {max_diff:.2e}")
print("PASS" if max_diff < 1e-4 else "FAIL")

In [ ]:
# Cell 11: Download
!zip -r deberta_l2.zip deberta_l2/

from google.colab import files
files.download("deberta_l2.zip")
print("\nExtract into parapet/models/deberta-l2/")